In [1]:
import numpy as np

npz_path = "/home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated/CKPT_SAVED/step_0100000/results_step0100000.npz"

def rel_l2(pred, true, eps=1e-12):
    pred = np.asarray(pred).reshape(-1)
    true = np.asarray(true).reshape(-1)
    num = np.linalg.norm(pred - true)
    den = np.linalg.norm(true) + eps
    return float(num / den)

with np.load(npz_path, allow_pickle=True) as z:
    keys = sorted(z.files)
    print(f"[npz] loaded: {npz_path}")
    print(f"[npz] #keys = {len(keys)}")
    print("[npz] key preview:", keys[:30], ("..." if len(keys) > 30 else ""))

    # -----------------------------
    # 1) Relative L2 error (overlay)
    # -----------------------------
    # 저장 코드 기준: ni_sim/ni_pinn, Te_sim/Te_pinn, E_sim/E_pinn, G_sim/G_pinn
    pairs = [
        ("ni_pinn", "ni_sim", "ni"),
        ("Te_pinn", "Te_sim", "Te"),
        ("E_pinn",  "E_sim",  "E"),
        ("G_pinn",  "G_sim",  "G"),
    ]

    print("\n[Relative L2 errors]")
    for pred_k, true_k, name in pairs:
        if pred_k in z and true_k in z:
            e = rel_l2(z[pred_k], z[true_k])
            print(f"  - {name:>2s}: relL2 = {e:.6e}   (using {pred_k} vs {true_k})")
        else:
            print(f"  - {name:>2s}: keys missing -> pred:{pred_k in z}, true:{true_k in z}")

    # -----------------------------
    # 2) Loss weights (SAW lambdas)
    # -----------------------------
    # log 저장 포맷: log__<colname>
    lam_keys = [
        "log__lambda_r1", "log__lambda_r2", "log__lambda_r3", "log__lambda_r4", "log__lambda_data",
        # 혹시 예전 호환으로 들어갔을 수도 있는 것들
        "log__lambda_phys", "log__lambda_bc",
    ]

    print("\n[Lambda (loss weight) availability]")
    for k in lam_keys:
        print(f"  - {k}: {'OK' if k in z else 'MISSING'}")

    # 마지막 값(가장 최근 로그 스텝) 출력
    # log__step이 있어야 "마지막 시점"을 제대로 특정 가능
    if "log__step" in z:
        steps = z["log__step"].astype(np.int64)
        last_i = int(np.argmax(steps))
        last_step = int(steps[last_i])
        print(f"\n[Last logged point] log__step = {last_step}, index = {last_i}")

        for k in lam_keys:
            if k in z:
                arr = np.asarray(z[k]).reshape(-1)
                if arr.size > last_i:
                    print(f"  - {k.replace('log__','')}: {float(arr[last_i]):.6e}")
                else:
                    print(f"  - {k.replace('log__','')}: present but shorter than steps (len={arr.size})")
    else:
        print("\n[warn] log__step not found. Printing last element of each lambda array if present.")
        for k in lam_keys:
            if k in z:
                arr = np.asarray(z[k]).reshape(-1)
                print(f"  - {k.replace('log__','')}: last = {float(arr[-1]):.6e} (len={arr.size})")

    # -----------------------------
    # (Optional) Loss terms too
    # -----------------------------
    loss_keys = [
        "log__total", "log__phys_total", "log__phys_r1", "log__phys_r2", "log__phys_r3", "log__phys_r4",
        "log__data_total"
    ]
    print("\n[Loss availability]")
    for k in loss_keys:
        print(f"  - {k}: {'OK' if k in z else 'MISSING'}")

[npz] loaded: /home/dlee/code/Plasma/Argon_steady_New_data_2510/Steady_BC_Hard_12_colloc0_make_gif_gradnorm_updated/CKPT_SAVED/step_0100000/results_step0100000.npz
[npz] #keys = 59
[npz] key preview: ['E_hat_grid', 'E_lab', 'E_pinn', 'E_sim', 'G_hat_grid', 'G_lab', 'G_pinn', 'G_sim', 'Te_hat_grid', 'Te_lab', 'Te_pinn', 'Te_sim', 'const__Lz', 'const__l0', 'const__t0', 'log__bc_dTe_L', 'log__bc_dTe_R', 'log__bc_flux_L', 'log__bc_flux_R', 'log__bc_pde_dTe_L', 'log__bc_pde_dTe_R', 'log__bc_pde_flux_L', 'log__bc_pde_flux_R', 'log__bc_total', 'log__columns', 'log__data_E', 'log__data_G', 'log__data_Te', 'log__data_n', 'log__data_total'] ...

[Relative L2 errors]
  - ni: relL2 = 3.336953e-03   (using ni_pinn vs ni_sim)
  - Te: relL2 = 3.067457e-04   (using Te_pinn vs Te_sim)
  -  E: relL2 = 6.085433e-02   (using E_pinn vs E_sim)
  -  G: relL2 = 4.490018e-03   (using G_pinn vs G_sim)

[Lambda (loss weight) availability]
  - log__lambda_r1: OK
  - log__lambda_r2: OK
  - log__lambda_r3: OK
  - l